In [2]:

import pandas as pd 
from pathlib import Path

In [3]:
from pathlib import Path

project_path = Path.home() / "Desktop" / "Supply Chain Disruption Intelligence"

print(project_path)

/Users/maitrayrangari/Desktop/Supply Chain Disruption Intelligence


In [4]:
print(project_path.exists())

True


In [ ]:
raw_path = project_path / 'data'/ 'raw'
print(raw_path)
print(raw_path.exists())

/Users/maitrayrangari/Desktop/Supply Chain Disruption Intelligence/Data/Raw
True


In [6]:
for file in raw_path.iterdir():
    print(file.name)

customer_master.csv
.DS_Store
product_master.csv
sales_orders.csv
supplier_master.csv
procurement_orders.csv
data_dictionary.csv


In [7]:
customer_df = pd.read_csv(raw_path / 'customer_master.csv')
product_df = pd.read_csv(raw_path / 'product_master.csv')
sales_df = pd.read_csv(raw_path / 'sales_orders.csv')
supplier_df = pd.read_csv(raw_path / 'supplier_master.csv')
procurement_df = pd.read_csv(raw_path / 'procurement_orders.csv')
dictionary_df = pd.read_csv(raw_path / 'data_dictionary.csv')

In [8]:
print("Customers:", customer_df.shape)
print("Products:", product_df.shape)
print("Suppliers:", supplier_df.shape)
print("Sales:", sales_df.shape)
print("Procurement:", procurement_df.shape)
print("Data Dictionary:", dictionary_df.shape)

Customers: (500, 8)
Products: (200, 10)
Suppliers: (100, 7)
Sales: (20000, 20)
Procurement: (2000, 9)
Data Dictionary: (54, 4)


In [9]:
sales_df.head()

,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Status,Order_Quantity,Unit_Price,Discount,Shipping_Mode,Shipping_Carrier,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
0,ORD0000001,CUST00481,PROD00135,2023-06-16,Completed,8,200.54,0.12,Standard,CarrierA,2023-06-21,2023-06-21,On Time,0,0.19,1510.96,176.4752,1411.8016,268.242304,-99.1584
1,ORD0000002,CUST00111,PROD00147,2023-09-23,Pending,3,220.96,0.18,Same Day,CarrierB,2023-09-26,2023-09-26,On Time,0,0.20,371.61,181.1872,543.5616,108.712320,171.9516
2,ORD0000003,CUST00104,PROD00172,2022-12-17,Cancelled,2,15.87,0.09,Economy,CarrierD,2022-12-18,2022-12-20,Late,1,0.20,16.46,14.4417,28.8834,5.776680,12.4234
3,ORD0000004,CUST00316,PROD00063,2023-10-11,Pending,7,191.91,0.06,Express,CarrierC,2023-10-15,2023-10-19,Late,1,0.00,1166.20,180.3954,1262.7678,0.000000,96.5678
4,ORD0000005,CUST00171,PROD00065,2022-10-04,Completed,6,192.00,0.19,Same Day,CarrierB,2022-10-05,2022-10-05,On Time,0,0.20,358.68,155.5200,933.1200,186.624000,574.4400


In [10]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Order_ID                 20000 non-null  str    
 1   Customer_ID              20000 non-null  str    
 2   Product_ID               20000 non-null  str    
 3   Order_Date               20000 non-null  str    
 4   Order_Status             20000 non-null  str    
 5   Order_Quantity           20000 non-null  int64  
 6   Unit_Price               20000 non-null  float64
 7   Discount                 20000 non-null  float64
 8   Shipping_Mode            20000 non-null  str    
 9   Shipping_Carrier         20000 non-null  str    
 10  Shipping_Date_Scheduled  20000 non-null  str    
 11  Shipping_Date_Actual     20000 non-null  str    
 12  Delivery_Status          20000 non-null  str    
 13  Late_Delivery_Risk_Flag  20000 non-null  int64  
 14  VAT_Rate                 20000 no

In [11]:
sales_df.isnull().sum()

Order_ID                   0
Customer_ID                0
Product_ID                 0
Order_Date                 0
Order_Status               0
Order_Quantity             0
Unit_Price                 0
Discount                   0
Shipping_Mode              0
Shipping_Carrier           0
Shipping_Date_Scheduled    0
Shipping_Date_Actual       0
Delivery_Status            0
Late_Delivery_Risk_Flag    0
VAT_Rate                   0
COGS                       0
Unit_Price_Effective       0
Order_Total                0
VAT_Amount                 0
Profit_Per_Order           0
dtype: int64

In [12]:
sales_df.duplicated().sum()

np.int64(0)

In [13]:
print("Total rows:", len(sales_df))
print("Unique order IDs:", sales_df["Order_ID"].nunique())

Total rows: 20000
Unique order IDs: 20000


In [14]:
## records where the actual shipping date is earlier than the scheduled date.

sales_df[
    sales_df["Shipping_Date_Actual"]
    < sales_df["Shipping_Date_Scheduled"]
].shape[0]

0

In [15]:
## orders where the actual shipping date is later than the scheduled date.

sales_df[
    sales_df["Shipping_Date_Actual"]
    > sales_df["Shipping_Date_Scheduled"]
].shape[0]

15915

In [16]:
## how many orders fall into each delivery-status category.

sales_df["Delivery_Status"].value_counts()

Delivery_Status
Late       15915
On Time     4085
Name: count, dtype: int64

In [17]:
## test whether the two conditions match row by row

late_by_shipping_date = (
    sales_df["Shipping_Date_Actual"]
    > sales_df["Shipping_Date_Scheduled"]
)

late_by_label = sales_df["Delivery_Status"] == "Late"

print("Both say late:", (late_by_shipping_date & late_by_label).sum())

print(
    "Date says late, label says on time:",
    (late_by_shipping_date & ~late_by_label).sum()
)

print(
    "Date says on time, label says late:",
    (~late_by_shipping_date & late_by_label).sum()
)

Both say late: 15915
Date says late, label says on time: 0
Date says on time, label says late: 0


In [18]:
## counts how many records have each risk-flag value.

sales_df['Late_Delivery_Risk_Flag'].value_counts()

Late_Delivery_Risk_Flag
1    15915
0     4085
Name: count, dtype: int64

In [19]:
## validate the risk flag against delivery status

sales_df[
    (sales_df["Delivery_Status"] == "Late")
    != (sales_df["Late_Delivery_Risk_Flag"] == 1)
].shape[0]

0

In [20]:
## how many orders are Completed, Pending, Cancelled, or any other status present.
sales_df["Order_Status"].value_counts()

Order_Status
Completed    16980
Pending       1998
Cancelled     1022
Name: count, dtype: int64

In [21]:
sales_df["Order_Quantity"].describe()

count    20000.000000
mean         5.010600
std          1.998421
min          1.000000
25%          4.000000
50%          5.000000
75%          6.000000
max         17.000000
Name: Order_Quantity, dtype: float64

In [22]:
sales_df.describe()

,Order_Quantity,Unit_Price,Discount,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,5.010600,177.064499,0.125237,0.795750,0.170097,495.331671,154.871285,775.661959,131.921423,280.330288
std,1.998421,70.748140,0.072164,0.403163,0.068044,366.048958,63.352813,458.933115,99.243639,242.946534
min,1.000000,15.870000,0.000000,0.000000,0.000000,6.080000,11.902500,12.537300,0.000000,-346.975000
25%,4.000000,128.990000,0.060000,1.000000,0.100000,202.300000,110.947400,435.438000,58.542611,88.357875
50%,5.000000,176.130000,0.130000,1.000000,0.200000,415.290000,154.261900,702.926750,114.065025,240.471750
75%,6.000000,230.440000,0.190000,1.000000,0.210000,719.840000,203.438200,1035.416800,186.550035,425.288000
max,17.000000,316.610000,0.250000,1.000000,0.250000,2641.800000,316.610000,4647.604500,1161.901125,2005.804500


In [23]:
dictionary_df.head(10)

,Table_Name,Column_Name,Description,Data_Type
0,product_master,Product_ID,Unique product identifier,string
1,product_master,SKU,Stock keeping unit code for inventory control,string
2,product_master,Product_Name,Descriptive product name,string
3,product_master,Category,High level product category,string
4,product_master,Subcategory,Subcategory within the main category,string
5,product_master,Unit,"Unit of measure (e.g., pcs)",string
6,product_master,Unit_Cost,Cost per unit (raw manufacturing cost),float
7,product_master,Standard_Price,Standard selling price per unit,float
8,product_master,Launch_Date,Date when product was introduced,date
9,product_master,Discontinuation_Date,Date when product is discontinued (if any),date


In [24]:
dictionary_df.columns

Index(['Table_Name', 'Column_Name', 'Description', 'Data_Type'], dtype='str')

In [25]:
dictionary_df[
    dictionary_df["Table_Name"].str.contains("sales", case=False, na=False)
]

,Table_Name,Column_Name,Description,Data_Type
25,sales_orders,Order_ID,Unique sales order identifier,string
26,sales_orders,Customer_ID,Customer associated with the order,string
27,sales_orders,Product_ID,Product ordered (foreign key to product_master),string
28,sales_orders,Order_Date,Date when the order was placed,date
29,sales_orders,Order_Status,"Status of the order (Completed, Cancelled, Pen...",string
30,sales_orders,Order_Quantity,Quantity of units ordered,integer
31,sales_orders,Unit_Price,Standard price per unit at time of order,float
32,sales_orders,Discount,Discount applied as a fraction of price,float
33,sales_orders,Shipping_Mode,"Mode of shipping (Standard, Express, etc.)",string
34,sales_orders,Shipping_Carrier,Carrier used for shipping,string


In [26]:
## disciunted unit price

expected_price = sales_df["Unit_Price"] * (1 - sales_df["Discount"])

(sales_df["Unit_Price_Effective"] - expected_price).abs().describe()

count    2.000000e+04
mean     3.652811e-15
std      8.755798e-15
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.842171e-14
dtype: float64

In [27]:
## total revenue after discount, excluding VAT

expected_total = (
    sales_df["Order_Quantity"]
    * sales_df["Unit_Price_Effective"]
)

(sales_df["Order_Total"] - expected_total).abs().describe()

count    2.000000e+04
mean     2.415668e-14
std      6.207471e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.821210e-13
dtype: float64

In [28]:
## Validate VAT_Amount

expected_vat = sales_df["Order_Total"] * sales_df["VAT_Rate"]

(sales_df["VAT_Amount"] - expected_vat).abs().describe()

count    2.000000e+04
mean     4.366207e-15
std      1.168920e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.273737e-13
dtype: float64

In [29]:
## Validate Profit_Per_Order

expected_profit = sales_df["Order_Total"] - sales_df["COGS"]

(sales_df["Profit_Per_Order"] - expected_profit).abs().describe()

count    2.000000e+04
mean     2.543884e-14
std      5.897702e-14
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.421085e-14
max      6.821210e-13
dtype: float64

In [30]:
## Check whether dates are stored correctly

sales_df[
    ["Order_Date", "Shipping_Date_Scheduled", "Shipping_Date_Actual"]
].head()

,Order_Date,Shipping_Date_Scheduled,Shipping_Date_Actual
0,2023-06-16,2023-06-21,2023-06-21
1,2023-09-23,2023-09-26,2023-09-26
2,2022-12-17,2022-12-18,2022-12-20
3,2023-10-11,2023-10-15,2023-10-19
4,2022-10-04,2022-10-05,2022-10-05


In [31]:
## Check the date data types

sales_df[
    ["Order_Date", "Shipping_Date_Scheduled", "Shipping_Date_Actual"]
].dtypes

Order_Date                 str
Shipping_Date_Scheduled    str
Shipping_Date_Actual       str
dtype: object

In [32]:
## Convert the three columns to datetime

date_columns = [
    "Order_Date",
    "Shipping_Date_Scheduled",
    "Shipping_Date_Actual"
]

for col in date_columns:
    sales_df[col] = pd.to_datetime(sales_df[col], errors="coerce")

In [33]:
sales_df[date_columns].dtypes

Order_Date                 datetime64[us]
Shipping_Date_Scheduled    datetime64[us]
Shipping_Date_Actual       datetime64[us]
dtype: object

In [34]:
## verify that none of the dates became missing.

sales_df[date_columns].isna().sum()

Order_Date                 0
Shipping_Date_Scheduled    0
Shipping_Date_Actual       0
dtype: int64

In [35]:
## Check the timeline for logical inconsistencies

sales_df[
    sales_df["Shipping_Date_Actual"] < sales_df["Order_Date"]
].shape[0]

0

In [36]:
## Check scheduled dates

sales_df[
    sales_df["Shipping_Date_Scheduled"] < sales_df["Order_Date"]
].shape[0]

0

In [37]:
## Measure the time between ordering and shipping

sales_df["Days_To_Ship"] = (
    sales_df["Shipping_Date_Actual"] - sales_df["Order_Date"]
).dt.days

sales_df["Days_To_Ship"].describe()

count    20000.000000
mean         5.475150
std          2.221808
min          1.000000
25%          4.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: Days_To_Ship, dtype: float64

In [38]:
## Compare shipping modes

sales_df.groupby("Shipping_Mode")["Days_To_Ship"].agg(
    ["count", "mean", "median"]
).round(2)

,count,mean,median
Shipping_Mode,,,
Economy,4972,5.48,6.0
Express,4979,5.45,5.0
Same Day,5033,5.49,6.0
Standard,5016,5.48,6.0


In [39]:
## Compare late-shipping rates by mode

sales_df.groupby("Shipping_Mode")["Delivery_Status"].apply(
    lambda x: (x == "Late").mean() * 100
).round(2)

Shipping_Mode
Economy     79.99
Express     79.63
Same Day    79.36
Standard    79.33
Name: Delivery_Status, dtype: float64

In [40]:
## Investigate the overall delay pattern

sales_df["Days_Late"] = (
    sales_df["Shipping_Date_Actual"]
    - sales_df["Shipping_Date_Scheduled"]
).dt.days

sales_df["Days_Late"].describe()

count    20000.000000
mean         1.981550
std          1.416407
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          4.000000
Name: Days_Late, dtype: float64

In [41]:
## How late are the delayed orders specifically?

sales_df.loc[
    sales_df["Days_Late"] > 0, "Days_Late"
].describe()

count    15915.000000
mean         2.490167
std          1.120075
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          4.000000
Name: Days_Late, dtype: float64

In [42]:
## Compare delay severity by shipping mode

sales_df.groupby("Shipping_Mode")["Days_Late"].agg(
    ["mean", "median"]
).round(2)

,mean,median
Shipping_Mode,,
Economy,2.00,2.0
Express,1.98,2.0
Same Day,1.97,2.0
Standard,1.98,2.0


In [43]:
## Check order status against delivery status

pd.crosstab(
    sales_df["Order_Status"],
    sales_df["Delivery_Status"]
)

Delivery_Status,Late,On Time
Order_Status,,
Cancelled,801,221
Completed,13526,3454
Pending,1588,410


In [44]:
## Check the delivery dates for cancelled orders

sales_df.loc[
    sales_df["Order_Status"] == "Cancelled",
    [
        "Order_ID",
        "Order_Date",
        "Shipping_Date_Scheduled",
        "Shipping_Date_Actual",
        "Delivery_Status"
    ]
].head(10)

,Order_ID,Order_Date,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status
2,ORD0000003,2022-12-17,2022-12-18,2022-12-20,Late
31,ORD0000032,2022-10-21,2022-10-22,2022-10-26,Late
42,ORD0000043,2023-09-14,2023-09-16,2023-09-19,Late
67,ORD0000068,2024-05-23,2024-05-26,2024-05-29,Late
109,ORD0000110,2023-04-01,2023-04-07,2023-04-09,Late
139,ORD0000140,2024-02-17,2024-02-23,2024-02-26,Late
216,ORD0000217,2024-05-12,2024-05-16,2024-05-20,Late
243,ORD0000244,2022-07-07,2022-07-09,2022-07-13,Late
251,ORD0000252,2022-11-07,2022-11-12,2022-11-15,Late
285,ORD0000286,2024-09-11,2024-09-12,2024-09-12,On Time


In [45]:
## Check whether this pattern exists for pending orders too

sales_df.loc[
    sales_df["Order_Status"] == "Pending",
    [
        "Order_ID",
        "Order_Date",
        "Shipping_Date_Scheduled",
        "Shipping_Date_Actual",
        "Delivery_Status"
    ]
].head(10)


,Order_ID,Order_Date,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status
1,ORD0000002,2023-09-23,2023-09-26,2023-09-26,On Time
3,ORD0000004,2023-10-11,2023-10-15,2023-10-19,Late
13,ORD0000014,2022-10-21,2022-10-25,2022-10-28,Late
18,ORD0000019,2022-03-09,2022-03-15,2022-03-15,On Time
22,ORD0000023,2023-06-08,2023-06-10,2023-06-12,Late
28,ORD0000029,2024-05-24,2024-05-30,2024-06-01,Late
40,ORD0000041,2024-01-15,2024-01-16,2024-01-20,Late
44,ORD0000045,2022-07-07,2022-07-12,2022-07-12,On Time
53,ORD0000054,2022-04-09,2022-04-11,2022-04-12,Late
54,ORD0000055,2022-01-21,2022-01-25,2022-01-26,Late


In [46]:
## quantify the issue

pd.crosstab(
    sales_df["Order_Status"],
    sales_df["Shipping_Date_Actual"].notna(),
    margins=True
)

Shipping_Date_Actual,True,All
Order_Status,,
Cancelled,1022,1022
Completed,16980,16980
Pending,1998,1998
All,20000,20000


In [47]:
## are the order statuses consistent with the shipping dates?

sales_df.groupby("Order_Status")["Days_To_Ship"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

,count,mean,median,min,max
Order_Status,,,,,
Cancelled,1022,5.39,5.0,1,10
Completed,16980,5.48,5.0,1,10
Pending,1998,5.47,5.0,1,10


In [48]:
## check whether the categorical columns contain unexpected values

categorical_columns = [
    "Order_Status",
    "Shipping_Mode",
    "Shipping_Carrier",
    "Delivery_Status"
]

for col in categorical_columns:
    print(f"\n{col}")
    print(sales_df[col].value_counts(dropna=False))


Order_Status
Order_Status
Completed    16980
Pending       1998
Cancelled     1022
Name: count, dtype: int64

Shipping_Mode
Shipping_Mode
Same Day    5033
Standard    5016
Express     4979
Economy     4972
Name: count, dtype: int64

Shipping_Carrier
Shipping_Carrier
CarrierA    4074
CarrierC    4057
CarrierD    4014
CarrierE    3939
CarrierB    3916
Name: count, dtype: int64

Delivery_Status
Delivery_Status
Late       15915
On Time     4085
Name: count, dtype: int64


In [49]:
## Check shipping carriers' performance

carrier_late_rate = (
    sales_df.groupby("Shipping_Carrier")["Delivery_Status"]
    .apply(lambda x: (x == "Late").mean() * 100)
    .sort_values(ascending=False)
    .round(2)
)

carrier_late_rate

Shipping_Carrier
CarrierE    80.65
CarrierD    80.29
CarrierA    79.82
CarrierC    78.78
CarrierB    78.32
Name: Delivery_Status, dtype: float64

In [50]:
## Compare carriers by shipping mode

pd.crosstab(
    sales_df["Shipping_Carrier"],
    sales_df["Shipping_Mode"],
    values=sales_df["Delivery_Status"].eq("Late").astype(int),
    aggfunc="mean"
).mul(100).round(2)

Shipping_Mode,Economy,Express,Same Day,Standard
Shipping_Carrier,,,,
CarrierA,79.45,80.53,80.80,78.52
CarrierB,80.06,77.11,77.34,78.70
CarrierC,80.17,78.08,77.49,79.51
CarrierD,81.38,82.50,78.94,78.34
CarrierE,78.92,79.74,82.36,81.61


In [51]:
## Check the sample sizes

pd.crosstab(
    sales_df["Shipping_Carrier"],
    sales_df["Shipping_Mode"]
)

Shipping_Mode,Economy,Express,Same Day,Standard
Shipping_Carrier,,,,
CarrierA,1056,1017,1000,1001
CarrierB,993,961,962,1000
CarrierC,963,981,1093,1020
CarrierD,983,1023,997,1011
CarrierE,977,997,981,984


In [52]:
## Check whether any text values are blank or inconsistent

text_columns = sales_df.select_dtypes(include="object").columns

for col in text_columns:
    blank_count = sales_df[col].astype("string").str.strip().eq("").sum()
    print(f"{col}: {blank_count} blank values")

Order_ID: 0 blank values
Customer_ID: 0 blank values
Product_ID: 0 blank values
Order_Status: 0 blank values
Shipping_Mode: 0 blank values
Shipping_Carrier: 0 blank values
Delivery_Status: 0 blank values


/var/folders/4y/19pdgt_n15xdphdkl2kbyqn00000gn/T/ipykernel_30940/2001492249.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = sales_df.select_dtypes(include="object").columns


In [53]:
text_columns = sales_df.select_dtypes(include=["object", "str"]).columns

for col in text_columns:
    blank_count = (
        sales_df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{col}: {blank_count} blank values")

Order_ID: 0 blank values
Customer_ID: 0 blank values
Product_ID: 0 blank values
Order_Status: 0 blank values
Shipping_Mode: 0 blank values
Shipping_Carrier: 0 blank values
Delivery_Status: 0 blank values


In [54]:
## Check the numeric columns for impossible values

sales_df.describe().T

,count,mean,min,25%,50%,75%,max,std
Order_Date,20000,2023-05-13 15:53:29.760000,2022-01-01 00:00:00,2022-09-02 00:00:00,2023-05-15 00:00:00,2024-01-20 00:00:00,2024-09-26 00:00:00,NaN
Order_Quantity,20000.0,5.0106,1.0,4.0,5.0,6.0,17.0,1.998421
Unit_Price,20000.0,177.064499,15.87,128.99,176.13,230.44,316.61,70.74814
Discount,20000.0,0.125237,0.0,0.06,0.13,0.19,0.25,0.072164
Shipping_Date_Scheduled,20000,2023-05-17 03:44:16.800000,2022-01-02 00:00:00,2022-09-06 00:00:00,2023-05-18 00:00:00,2024-01-24 00:00:00,2024-10-02 00:00:00,NaN
Shipping_Date_Actual,20000,2023-05-19 03:17:42.720000,2022-01-03 00:00:00,2022-09-08 00:00:00,2023-05-20 00:00:00,2024-01-26 00:00:00,2024-10-06 00:00:00,NaN
Late_Delivery_Risk_Flag,20000.0,0.79575,0.0,1.0,1.0,1.0,1.0,0.403163
VAT_Rate,20000.0,0.170097,0.0,0.1,0.2,0.21,0.25,0.068044
COGS,20000.0,495.331671,6.08,202.3,415.29,719.84,2641.8,366.048958
Unit_Price_Effective,20000.0,154.871285,11.9025,110.9474,154.2619,203.4382,316.61,63.352813


In [55]:
datasets = {
    "Customers": customer_df,
    "Products": product_df,
    "Suppliers": supplier_df,
    "Procurement": procurement_df
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Columns:", df.columns.tolist())


--- Customers ---
Shape: (500, 8)
Missing values: 0
Duplicate rows: 0
Columns: ['Customer_ID', 'Customer_Name', 'Industry', 'Market_Segment', 'Country', 'City', 'Latitude', 'Longitude']

--- Products ---
Shape: (200, 10)
Missing values: 200
Duplicate rows: 0
Columns: ['Product_ID', 'SKU', 'Product_Name', 'Category', 'Subcategory', 'Unit', 'Unit_Cost', 'Standard_Price', 'Launch_Date', 'Discontinuation_Date']

--- Suppliers ---
Shape: (100, 7)
Missing values: 21
Duplicate rows: 0
Columns: ['Supplier_ID', 'Supplier_Name', 'Country', 'Region', 'On_Time_Delivery_Rate', 'Certification_Level', 'Preferred_Supplier_Flag']

--- Procurement ---
Shape: (2000, 9)
Missing values: 0
Duplicate rows: 0
Columns: ['PO_ID', 'Supplier_ID', 'Raw_Material_ID', 'Order_Date', 'Order_Quantity', 'Unit_Cost', 'Delivery_Date_Planned', 'Delivery_Date_Actual', 'Total_Cost']


In [56]:
## Identify the missing values

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n--- {name} ---")
        print(missing)


--- Products ---
Discontinuation_Date    200
dtype: int64

--- Suppliers ---
Certification_Level    21
dtype: int64


In [57]:
## Check primary keys

key_columns = {
    "Customers": (customer_df, "Customer_ID"),
    "Products": (product_df, "Product_ID"),
    "Suppliers": (supplier_df, "Supplier_ID"),
    "Sales": (sales_df, "Order_ID"),
    "Procurement": (procurement_df, "PO_ID")
}

for name, (df, key) in key_columns.items():
    print(
        f"{name}: {df[key].nunique()} unique IDs "
        f"out of {len(df)} rows"
    )

Customers: 500 unique IDs out of 500 rows
Products: 200 unique IDs out of 200 rows
Suppliers: 100 unique IDs out of 100 rows
Sales: 20000 unique IDs out of 20000 rows
Procurement: 2000 unique IDs out of 2000 rows


In [58]:
## Relationships between tables

checks = {
    "Sales → Customers": (
        sales_df["Customer_ID"],
        customer_df["Customer_ID"]
    ),
    "Sales → Products": (
        sales_df["Product_ID"],
        product_df["Product_ID"]
    ),
    "Procurement → Suppliers": (
        procurement_df["Supplier_ID"],
        supplier_df["Supplier_ID"]
    )
}

for name, (child_ids, parent_ids) in checks.items():
    unmatched = (~child_ids.isin(parent_ids)).sum()
    print(f"{name}: {unmatched} unmatched IDs")

Sales → Customers: 0 unmatched IDs
Sales → Products: 0 unmatched IDs
Procurement → Suppliers: 0 unmatched IDs


In [60]:
## separate DataFrames so we can make changes without modifying the original loaded DataFrames.

# Create working copies of the raw datasets
customers_clean = customer_df.copy()
products_clean = product_df.copy()
suppliers_clean = supplier_df.copy()
sales_clean = sales_df.copy()
procurement_clean = procurement_df.copy()

In [61]:
customers_clean.shape

(500, 8)